# Aggregating, Reshaping and Combining DataFrames

## Introduction to Aggregation, Reshaping and Combining

There are a number of functions:
- **Aggregation** calculates the summary of each group.
- **Reshaping** rearanges the representation but doesn`t nessesarily calculate a summary.
- **Combining** stacks records/matches them using a key; it brings seperate records together.

**Common Aggregation Functions**
- ***sum():*** Calculates total of numeric values.
- ***mean():*** Calculates arithmetic average.
- ***median():*** Middle value after sorting.
- ***min() & max():*** Smallest and largest value.
- ***count():*** Counts present values in a chosen field.
- ***nunique():*** Number of distinct values.
- ***size():*** Numbers of rows in a column.

## Process for aggregating, reshaping and combining dataframes
- Choose the grouping columns and the value to surmarise.
- Select an aggregation function that matches the question and units.
- Report the missing values/observation counts alongside summaries.
- Reshape only when the new arrangement helps compare values.
- For a merge, identify the common key and expected relationship.
- Validate output row counts, unmatched keys and totals before reporting.



In [48]:
import pandas as pd
clinics_visits = pd.read_csv("C:\\Users\\mish\\Desktop\\git_practice\\clinic_visits_cleaned.csv")
clinics_visits.head()

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes
0,V001,P01,C01,2026-01-05,24.0,consultation,1200.0,30.0
1,V002,P02,C02,2026-01-05,35.0,LAB,800.0,45.0
2,V004,P04,C03,2026-01-06,42.0,Pharmacy,500.0,15.0
3,V005,P05,C02,2026-02-03,29.0,consultation,1200.0,35.0
4,V006,P01,C01,2026-02-03,24.0,Lab,800.0,35.0


In [49]:
clinics_visits_cleaned = clinics_visits.copy()
clinics_visits_cleaned .head()

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes
0,V001,P01,C01,2026-01-05,24.0,consultation,1200.0,30.0
1,V002,P02,C02,2026-01-05,35.0,LAB,800.0,45.0
2,V004,P04,C03,2026-01-06,42.0,Pharmacy,500.0,15.0
3,V005,P05,C02,2026-02-03,29.0,consultation,1200.0,35.0
4,V006,P01,C01,2026-02-03,24.0,Lab,800.0,35.0


## Questions for aggregation summaries
1. Check for the aggregation summaries.
2. Group by any cartegory.
3. Group by 2 columns.

In [50]:
# Question 1: Calculate average wait time(mean), fees(mean and sum), unique ids(unique)
clinics_visits_cleaned.describe()

,age,fee_kes,wait_minutes
count,10.000000,9.000000,10.000000
mean,37.800000,911.111111,35.000000
std,11.659998,297.676185,15.092309
min,24.000000,500.000000,10.000000
25%,29.500000,800.000000,30.000000
50%,36.500000,800.000000,35.000000
75%,44.250000,1200.000000,43.750000
max,60.000000,1200.000000,60.000000


In [51]:
# Question 2: Group by any category
clinics_visits_cleaned.groupby('clinic_id')[ 'fee_kes'].describe()

,count,mean,std,min,25%,50%,75%,max
clinic_id,,,,,,,,
C01,2.0,1000.000000,282.842712,800.0,900.0,1000.0,1100.0,1200.0
C02,4.0,1000.000000,230.940108,800.0,800.0,1000.0,1200.0,1200.0
C03,3.0,733.333333,404.145188,500.0,500.0,500.0,850.0,1200.0


In [53]:
# Question 3: Group by 2 columns
clinics_visits_cleaned.groupby(['clinic_id', 'patient_id'])['fee_kes'].describe()

count    mean         std     min     25%     50%  \
clinic_id patient_id                                                      
C01       P01           2.0  1000.0  282.842712   800.0   900.0  1000.0   
          P08           0.0     NaN         NaN     NaN     NaN     NaN   
C02       P02           1.0   800.0         NaN   800.0   800.0   800.0   
          P05           1.0  1200.0         NaN  1200.0  1200.0  1200.0   
          P07           1.0  1200.0         NaN  1200.0  1200.0  1200.0   
          P10           1.0   800.0         NaN   800.0   800.0   800.0   
C03       P04           1.0   500.0         NaN   500.0   500.0   500.0   
          P09           1.0   500.0         NaN   500.0   500.0   500.0   
          P11           1.0  1200.0         NaN  1200.0  1200.0  1200.0   

                         75%     max  
clinic_id patient_id                  
C01       P01         1100.0  1200.0  
          P08            NaN     NaN  
C02       P02          800.0   800.0  
          P05         1200.0  1200.0  
          P07         1200.0  1200.0  
          P10          800.0   800.0  
C03       P04          500.0   500.0  
          P09          500.0   500.0  
          P11         1200.0  1200.0

## Questions for reshaping 
1. Explain and create a pivot table.
2. Return a wide table to longform using melt.
- Wide form spreads categories across columns e.g. service_grid.
- Longform stores category manes in one column and stores values in another 

 

In [55]:
clinic_pivot = clinics_visits_cleaned.pivot_table(
    values='fee_kes',
    index= 'patient_id',
    columns= 'clinic_id',
    aggfunc='sum')
(clinic_pivot)

clinic_id,C01,C02,C03
patient_id,,,
P01,2000.0,NaN,NaN
P02,NaN,800.0,NaN
P04,NaN,NaN,500.0
P05,NaN,1200.0,NaN
P07,NaN,1200.0,NaN
P08,0.0,NaN,NaN
P09,NaN,NaN,500.0
P10,NaN,800.0,NaN
P11,NaN,NaN,1200.0


In [56]:
longform_clinic_visit = clinic_pivot.reset_index(names='patient_id').melt(id_vars=['patient_id'], var_name='clinic_id', value_name= 'fee_kes')
longform_clinic_visit

,patient_id,clinic_id,fee_kes
0,P01,C01,2000.0
1,P02,C01,NaN
2,P04,C01,NaN
3,P05,C01,NaN
4,P07,C01,NaN
5,P08,C01,0.0
6,P09,C01,NaN
7,P10,C01,NaN
8,P11,C01,NaN
9,P01,C02,NaN


## Questions for Combining dataframes 
1. Merge tables using a common key(use the clinc_visits_raw.csv and clinics.csv)
2. Display clinics with no visits.
3. Concatinate the match.csv file to the most appropriate csv file.
4. Extract date components and add them as new columns.
 

In [57]:
clinic_visits_raw = pd.read_csv("clinic_visits_raw.csv")
clinics_names = pd.read_csv('C:\\Users\\mish\\Desktop\\git_practice\\clinic_visits_cleaned.csv')
merged_clinic_data = pd.merge(clinic_visits_raw,clinics_names, on='clinic_id', how='inner')


In [58]:
merged_clinic_data

,visit_id_x,patient_id_x,clinic_id,visit_date_x,age_x,service_x,fee_kes_x,wait_minutes_x,visit_id_y,patient_id_y,visit_date_y,age_y,service_y,fee_kes_y,wait_minutes_y
0,V001,P01,C01,2026-01-05,24.0,consultation,1200,30.0,V001,P01,2026-01-05,24.0,consultation,1200.0,30.0
1,V001,P01,C01,2026-01-05,24.0,consultation,1200,30.0,V006,P01,2026-02-03,24.0,Lab,800.0,35.0
2,V001,P01,C01,2026-01-05,24.0,consultation,1200,30.0,V009,P08,2026-02-05,50.0,Lab,NaN,50.0
3,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0,V002,P02,2026-01-05,35.0,LAB,800.0,45.0
4,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0,V005,P05,2026-02-03,29.0,consultation,1200.0,35.0
5,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0,V008,P07,2026-02-04,31.0,Consultation,1200.0,40.0
6,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0,V011,P10,NaN,45.0,Lab,800.0,30.0
7,V003,P03,C01,2026-01-06,NaN,Consultation,1200,25.0,V001,P01,2026-01-05,24.0,consultation,1200.0,30.0
8,V003,P03,C01,2026-01-06,NaN,Consultation,1200,25.0,V006,P01,2026-02-03,24.0,Lab,800.0,35.0
9,V003,P03,C01,2026-01-06,NaN,Consultation,1200,25.0,V009,P08,2026-02-05,50.0,Lab,NaN,50.0


In [59]:
clinics_names.set_index('clinic_id', inplace=True)

In [62]:
merged_clinic_data_2 = clinic_visits_raw.join(
	clinics_names,
	on='clinic_id',
	how='inner',
	lsuffix='_raw',
	rsuffix='_cleaned'
)
merged_clinic_data_2

,visit_id_raw,patient_id_raw,clinic_id,visit_date_raw,age_raw,service_raw,fee_kes_raw,wait_minutes_raw,visit_id_cleaned,patient_id_cleaned,visit_date_cleaned,age_cleaned,service_cleaned,fee_kes_cleaned,wait_minutes_cleaned
0,V001,P01,C01,2026-01-05,24.0,consultation,1200,30.0,V001,P01,2026-01-05,24.0,consultation,1200.0,30.0
0,V001,P01,C01,2026-01-05,24.0,consultation,1200,30.0,V006,P01,2026-02-03,24.0,Lab,800.0,35.0
0,V001,P01,C01,2026-01-05,24.0,consultation,1200,30.0,V009,P08,2026-02-05,50.0,Lab,NaN,50.0
1,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0,V002,P02,2026-01-05,35.0,LAB,800.0,45.0
1,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0,V005,P05,2026-02-03,29.0,consultation,1200.0,35.0
1,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0,V008,P07,2026-02-04,31.0,Consultation,1200.0,40.0
1,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0,V011,P10,NaN,45.0,Lab,800.0,30.0
2,V003,P03,C01,2026-01-06,NaN,Consultation,1200,25.0,V001,P01,2026-01-05,24.0,consultation,1200.0,30.0
2,V003,P03,C01,2026-01-06,NaN,Consultation,1200,25.0,V006,P01,2026-02-03,24.0,Lab,800.0,35.0
2,V003,P03,C01,2026-01-06,NaN,Consultation,1200,25.0,V009,P08,2026-02-05,50.0,Lab,NaN,50.0


In [64]:
march_visits = pd.read_csv('C:\\Users\\mish\\Desktop\\git_practice\\march_visits.csv')
merged_clinic_data_3 = pd.concat([clinics_visits_cleaned, march_visits], ignore_index=True)


In [66]:
merged_clinic_data_3['visit_date'] = pd.to_datetime(merged_clinic_data_3['visit_date'])
merged_clinic_data_3['year'] = merged_clinic_data_3['visit_date'].dt.year
merged_clinic_data_3['month'] = merged_clinic_data_3['visit_date'].dt.month_name()
merged_clinic_data_3['day_of_the_week'] = merged_clinic_data_3['visit_date'].dt.day_name()
merged_clinic_data_3

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes,year,month,day_of_the_week
0,V001,P01,C01,2026-01-05,24.0,consultation,1200.0,30.0,2026.0,January,Monday
1,V002,P02,C02,2026-01-05,35.0,LAB,800.0,45.0,2026.0,January,Monday
2,V004,P04,C03,2026-01-06,42.0,Pharmacy,500.0,15.0,2026.0,January,Tuesday
3,V005,P05,C02,2026-02-03,29.0,consultation,1200.0,35.0,2026.0,February,Tuesday
4,V006,P01,C01,2026-02-03,24.0,Lab,800.0,35.0,2026.0,February,Tuesday
5,V008,P07,C02,2026-02-04,31.0,Consultation,1200.0,40.0,2026.0,February,Wednesday
6,V009,P08,C01,2026-02-05,50.0,Lab,NaN,50.0,2026.0,February,Thursday
7,V010,P09,C03,2026-02-05,38.0,Pharmacy,500.0,10.0,2026.0,February,Thursday
8,V011,P10,C02,NaT,45.0,Lab,800.0,30.0,NaN,NaN,NaN
9,V012,P11,C03,2026-02-06,60.0,Consultation,1200.0,60.0,2026.0,February,Friday
